In [ ]:
!pip3 install -U ucimlrepo

from ucimlrepo import fetch_ucirepo

# fetch dataset
rt_iot2022 = fetch_ucirepo(id=942)

# data (as pandas dataframes)
X = rt_iot2022.data.features
y = rt_iot2022.data.targets

# metadata
print(rt_iot2022.metadata)

# variable information
print(rt_iot2022.variables)

In [ ]:
print("Headers for X:")
print(X.columns.tolist())
print("\nHeaders for y:")
print(y.columns.tolist())

1.	Data Cleaning

Before proceeding with further data processing, it is crucial to confirm the absence of missing values in the dataset. Although the dataset metadata may indicate that there are no missing values, a thorough verification is necessary to ensure data integrity. This step involves systematically checking the dataset to validate that all entries are complete and no missing values are present. By performing this verification, potential issues related to data completeness can be proactively addressed, ensuring the reliability of subsequent analyses.


In [ ]:
import pandas as pd

missing_values_count = X.isnull().sum()
missing_values_percentage = (missing_values_count / len(X)) * 100

missing_info = pd.DataFrame({
    'Missing Count': missing_values_count,
    'Missing Percentage': missing_values_percentage
})

# Filter to show only columns with missing values
missing_info = missing_info[missing_info['Missing Count'] > 0]

if missing_info.empty:
    print("No missing values found in the feature dataset X.")
else:
    print("Columns with missing values in X:")
    display(missing_info)

2.	Removing Duplicate Rows

Next, I will identify and remove any duplicate rows in the feature dataset X.


In [ ]:
initial_rows = X.shape[0]
duplicate_rows = X.duplicated().sum()

if duplicate_rows > 0:
    X.drop_duplicates(inplace=True)
    final_rows = X.shape[0]
    print(f"Removed {duplicate_rows} duplicate rows from X. \nInitial rows: {initial_rows}, Final rows: {final_rows}.")
else:
    print("No duplicate rows found in the feature dataset X.")

3.	Initial Exploration and Addressing Outliers

Identifying outliers can be challenging and often demands expertise in the subject area. A helpful initial approach is to examine the descriptive statistics for the numerical features in X. This analysis offers insights into their range, average values, and variability—details that can suggest where outliers might exist. For a more thorough investigation, methods such as the interquartile range (IQR), Z-score calculations, or visual tools like box plots are commonly applied to specific features uncovered during exploratory data analysis.


In [ ]:
numerical_cols = X.select_dtypes(include=['number']).columns
print("Descriptive statistics for numerical features in X:")
display(X[numerical_cols].describe())

**Exploratory Data Analysis (EDA)**

In [ ]:
print("Information about feature dataset X:")
X.info()

print("\nInformation about target dataset y:")
y.info()

1.	Understand the Target Variable Distribution

To assess class imbalance, I need to examine the target variable, Attack_Type.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.countplot(data=y, x='Attack_type', palette='viridis')
plt.title('Distribution of Attack Types')
plt.xlabel('Attack Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**2.	Distribution of Categorical Features – Univariate Analysis for Categorical Features in X**

This process examines each feature separately, allowing the data to assess its distribution and key measures such as the mean, median, variance, and standard deviation.


**Distribution of Numerical Features**

Let's visualize the distributions of some numerical features using histograms to understand their spread and detect potential outliers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

initial_rows = X.shape[0]
duplicate_rows = X.duplicated().sum()

if duplicate_rows > 0:
    X = X.drop_duplicates() # Modified line to avoid SettingWithCopyWarning
    final_rows = X.shape[0]
    print(f"Removed {duplicate_rows} duplicate rows from X. \nInitial rows: {initial_rows}, Final rows: {final_rows}.")
else:
    print("No duplicate rows found in the feature dataset X.")

numerical_cols = X.select_dtypes(include=['number']).columns

# For better visualization, let's select a subset of numerical columns
# and avoid plotting columns with very high variance or constant values.
# We will pick the first few and some that might be interesting from the metadata.

selected_numerical_cols = ['flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_pkts_per_sec', 'flow_pkts_per_sec', 'payload_bytes_per_second', 'active.avg', 'idle.avg']

### Distribution of Categorical Features

Let's examine the distribution of categorical features using count plots.

In [ ]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns

if not categorical_cols.empty:
    plt.figure(figsize=(16, 6 * len(categorical_cols)))
    for i, col in enumerate(categorical_cols):
        plt.subplot(len(categorical_cols), 1, i + 1)
        sns.countplot(data=X, x=col, palette='viridis')
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No categorical features found in X.")

3.	Aligning Target Variable y with Feature Set X

Input Code:

After removing duplicate rows from X, the index of X has changed, and its number of rows is now different from the original y. For consistent analysis and model building, y must be aligned with the filtered X. We will re-index y to match the current index of X.


This process examines each feature separately, allowing the data to assess its distribution and key measures such as the mean, median, variance, and standard deviation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

categorical_cols = X.select_dtypes(include=['object', 'category']).columns

if not categorical_cols.empty:
    plt.figure(figsize=(16, 6 * len(categorical_cols)))
    for i, col in enumerate(categorical_cols):
        plt.subplot(len(categorical_cols), 1, i + 1)
        sns.countplot(data=X, x=col, palette='viridis')
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No categorical features found in X.")

The output shows that the highest count is for the TCP protocol.

In [ ]:
# Align y with the current index of X after duplicate removal
y = y.loc[X.index]

print(f"Number of rows in X after duplicate removal: {X.shape[0]}")
print(f"Number of rows in y after alignment: {y.shape[0]}")

if X.shape[0] == y.shape[0]:
    print("X and y are now aligned with the same number of rows.")
else:
    print("Error: X and y still have a different number of rows.")

**Bivariate Analysis: Features vs. Target Variable Attack_type**

Now, let's explore the relationships between some key features in X and the target variable y ('Attack_type'). This will help us understand which features might be most indicative of different attack types.
This also involves examining the relationship between two variables. It helps me identify associations, correlations, and differences between groups.


In [ ]:
import numpy as np

# Define a small epsilon to avoid division by zero
epsilon = 1e-6

# 1. Total Packets and Bytes
X['Total_Packets'] = X['fwd_pkts_tot'] + X['bwd_pkts_tot']
X['Total_Bytes_Payload'] = X['fwd_pkts_payload.tot'] + X['bwd_pkts_payload.tot']

# 2. Packet Ratios (handling division by zero)
X['Fwd_Bwd_Packet_Ratio'] = X['fwd_pkts_tot'] / (X['bwd_pkts_tot'] + epsilon)
X['Fwd_Bwd_Data_Packet_Ratio'] = X['fwd_data_pkts_tot'] / (X['bwd_data_pkts_tot'] + epsilon)

# 3. Average Packet Sizes (handling division by zero)
X['Avg_Fwd_Packet_Size'] = X['fwd_pkts_payload.tot'] / (X['fwd_pkts_tot'] + epsilon)
X['Avg_Bwd_Packet_Size'] = X['bwd_pkts_payload.tot'] / (X['bwd_pkts_tot'] + epsilon)

# 4. Flag Rates (relative to total packets, handling division by zero)
X['SYN_Flag_Rate'] = X['flow_SYN_flag_count'] / (X['Total_Packets'] + epsilon)
X['FIN_Flag_Rate'] = X['flow_FIN_flag_count'] / (X['Total_Packets'] + epsilon)
X['RST_Flag_Rate'] = X['flow_RST_flag_count'] / (X['Total_Packets'] + epsilon)

print("X after adding new domain features:")
display(X.head())
print(f"New shape of X: {X.shape}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure 'flow_duration' is treated as numerical for binning
# (it should already be after RobustScaler, but good to be explicit if types were mixed)
X['flow_duration'] = pd.to_numeric(X['flow_duration'], errors='coerce')

# Drop rows where 'flow_duration' might have become NaN due to coercion issues if any (unlikely here)
X.dropna(subset=['flow_duration'], inplace=True)

# Apply equal-frequency binning to 'flow_duration'
# Using qcut ensures each bin has approximately the same number of observations
# You can adjust the number of bins (q=5) as needed.
X['flow_duration_binned'] = pd.qcut(X['flow_duration'], q=5, labels=False, duplicates='drop')

print("Distribution of 'flow_duration_binned' after binning:")
display(X['flow_duration_binned'].value_counts().sort_index())

# Visualize the distribution of the binned column
plt.figure(figsize=(8, 5))
sns.countplot(x='flow_duration_binned', data=X, palette='viridis')
plt.title('Distribution of Binned Flow Duration')
plt.xlabel('Flow Duration Bin')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Display the head of X to show the new column
print("\nX with new 'flow_duration_binned' column:")
display(X.head())

**Initial Inference:**

**Univariate Analysis:**

•	**Attack_type Distribution:**
The data shows marked class imbalance, with some attack types (e.g., 'DOS_SYN_Hping', 'MQTT_Publish') occurring far more frequently—an important factor for model development due to potential bias.

•	**Categorical Features (proto, service): **
Protocols like 'tcp' dominate network traffic, and the '-' service category is especially frequent.

**Bivariate Analysis:**

**•	Numerical Features vs. Attack_type:**
Box plots reveal that features such as flow_duration, fwd_pkts_per_sec, and payload_bytes_per_second vary noticeably across attack types, with outliers indicating anomalous behavior.

**•	Categorical Features vs. Attack_type:**
Certain attack types are closely linked to specific protocols and services, e.g., 'DOS_SYN_Hping' mostly uses 'tcp', while 'MQTT_Publish' heavily involves the 'mqtt' service.

**Conclusion:**

Attack_type shows clear patterns across both numerical and categorical features, highlighting their value for classification. Addressing class imbalance during model training will be crucial to prevent biased results.
